# Evaluation Metrics for Generative Modeling on Spatial Transcriptomics
**Data:** paired spatial-transcriptomics slides (Zhuang ABCA mouse-brain atlas).

A runnable companion to the `paired_slides_eval` suite. The notebook follows one linear workflow:

1. **§1 Generate** the cells to evaluate (per model → a `generated.h5ad`).
2. **§2 Build the shared apparatus** — load the **source / target / classifier** slides and fit
   them into *one* whitened-PCA(50) basis + standardised-coord frame (the NicheFlow recipe).
3. **§3 Train probes** — the cell-type classifier for `ct/*`, and the masked expression regressor
   for `recon/*`.
4. **§4 Evaluate** — run the whole suite and write per-model CSVs (**§4a**), or run a single metric
   group and just print it (**§4b**).

Everything is scored through the **same** apparatus, so metrics are comparable across models: a
model that emits gene-space cells + raw coords (OT-CFM) is projected/standardised into the exact
basis the niche models (NicheFlow CFM/VFM) already live in — handled automatically by
`evaluate_files(..., coords="auto")`. All paths below are **relative to `notebooks/`**.

## 0. Environment setup

Per-project `uv` venv (run from the `eval_metrics/` repo root):

In [5]:
%%bash
UV_LINK_MODE=copy uv sync --extra pipeline --extra nicheflow --extra classifier 

Resolved 268 packages in 2ms
Checked 245 packages in 60ms


In [8]:
%%bash
cd /work/magroup/shared/enfuzers/generative_modeling/eval_metrics
env -u VIRTUAL_ENV uv run python -m ipykernel install --user \
    --name eval_metrics --display-name "Python (eval_metrics)"

Installed kernelspec eval_metrics in /home/qimow/.local/share/jupyter/kernels/eval_metrics


In [1]:
import sys, torch
print(sys.executable)
print(torch.__version__, torch.version.cuda)

/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/bin/python3
2.12.1+cu126 12.6


In [1]:
import importlib

import numpy as np
import pandas as pd


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


print("optional backends present:")
print("  torch   (MMD / EMD / classifier):", _has("torch"))
print("  ot/pot  (EMD / Wasserstein)     :", _has("ot"))
print("  squidpy (Moran's I)             :", _has("squidpy"))

optional backends present:
  torch   (MMD / EMD / classifier): True
  ot/pot  (EMD / Wasserstein)     : True
  squidpy (Moran's I)             : False


## 1. Generate the cells to evaluate

The metrics compare the real **target slide** against a model's **generated cells**. Generation is
model-specific and already done for the four models below — each stores its cells as a flat/niche
`.h5ad`. The cell just confirms the files are present.

| key | emits | needs shared-PCA replay? |
|---|---|---|
| `otcfm` / `otcfm_spatial` | flat **gene-space** cells + **raw** coords | yes (`shared_pca` + `standardize_coords`) |
| `nicheflow_cfm` / `nicheflow_vfm` | already **whitened X_pca(50)** + **standardised** coords | no (pass-through) |

The next cell just checks the files exist; the one after it **regenerates** a model on the
allocated GPU (run it only if you want fresh cells — it overwrites and needs a checkpoint).

In [ ]:
import os

# Each model's generated cells (one flat/niche .h5ad per model).
ARTIFACTS = {
    "otcfm":          "../artifacts/otcfm_1025/generated.h5ad",               # gene-space + raw coords
    "otcfm_spatial":  "../artifacts/otcfm_spatial_1025/generated.h5ad",       # gene-space + raw coords
    "nicheflow_cfm":  "../artifacts/nicheflow_cfm_unaligned/generated.h5ad",  # whitened X_pca + std coords
    "nicheflow_vfm":  "../artifacts/nicheflow_vfm/generated.h5ad",            # whitened X_pca + std coords
}
for name, path in ARTIFACTS.items():
    print(f"{'OK     ' if os.path.exists(path) else 'MISSING'}  {name:14s} {path}")

In [ ]:
%%bash
# OPTIONAL — (re)generate NicheFlow CFM cells on the allocated GPU (overwrites the artifact; needs a
# checkpoint). The kernel runs inside your SLURM job, so this uses the job's GPU automatically.
# `cd ..` -> repo root, so the paths match what you'd type in a terminal there.
cd ..
python -m paired_slides_eval.generate generator=nicheflow \
    source=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
    target=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    checkpoint=ckpts/NicheFlow_CFM_ABCA.ckpt \
    generated_out=artifacts/nicheflow_cfm_unaligned/generated.h5ad

## 2. Build the shared evaluation apparatus (source · target · classifier slide)

One `preprocess_pair` fit defines the whole measuring apparatus and is reused everywhere:

- **Expression:** `normalize_total → log1p → shared PCA (fit on source+target) → whiten`.
- **Coordinates:** per-slide standardisation `(pos − mean) / std`.

It produces two pickles that carry the *persisted recipe* (`lognorm_mean`, `lognorm_target_sum`,
`var_names`, `PCs`, `stats['X_pca']`, `stats['coords']`):

| file | what it is |
|---|---|
| `../data/abca_pair.pkl` | the apparatus **and** the eval target — source `1.024` → target `1.025` |
| `../data/abca_1.026_clf.pkl` | the held-out **classifier** slide `1.026` projected into the *same* basis |

Slide identities for this dataset: **target = `1.025`** (what every model is scored against),
**generation pair = `1.024 → 1.025`**, **classifier slide = `1.026`** (held out — neither source
nor target). Equivalent one-liner CLI:

```bash
python -m paired_slides_eval.adapters.prepare_shared_slides \
  --source ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
  --target ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
  --classifier_slide ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.026.h5ad \
  --ct_key class --n_pcs 50 \
  --out_pair ../data/abca_pair.pkl --out_classifier ../data/abca_1.026_clf.pkl
```

In [ ]:
import os
import pickle

from paired_slides_eval.adapters.nicheflow.preprocess import (
    preprocess_classifier_slide,
    preprocess_pair,
)

# --- raw input slides (AnnData .h5ad: raw genes in X + obsm['spatial'] + obs['class']) ---
SLIDES          = "../../nicheflow_mba/data"
SOURCE_H5AD     = f"{SLIDES}/adata_Zhuang_Zhuang-ABCA-1.024.h5ad"   # source (A)
TARGET_H5AD     = f"{SLIDES}/adata_Zhuang_Zhuang-ABCA-1.025.h5ad"   # target (B) = eval target
CLASSIFIER_H5AD = f"{SLIDES}/adata_Zhuang_Zhuang-ABCA-1.026.h5ad"   # held-out classifier-training slide

# --- shared artifacts written here (one fit -> one PCA basis + coord frame) ---
PAIR_PKL = "../data/abca_pair.pkl"         # apparatus + eval target (slide B)
CLF_PKL  = "../data/abca_1.026_clf.pkl"    # classifier slide (1.026) in the same basis
CT_KEY, N_PCS = "class", 50

os.makedirs("../data", exist_ok=True)
if os.path.exists(PAIR_PKL) and os.path.exists(CLF_PKL):
    print(f"using existing {PAIR_PKL}\n                {CLF_PKL}")
else:
    ds_pair, pre = preprocess_pair(SOURCE_H5AD, TARGET_H5AD, n_pcs=N_PCS, cell_type_column=CT_KEY)
    clf_ds = preprocess_classifier_slide(CLASSIFIER_H5AD, pre, cell_type_column=CT_KEY)
    pickle.dump(ds_pair, open(PAIR_PKL, "wb"))
    pickle.dump(clf_ds, open(CLF_PKL, "wb"))
    print(f"built {PAIR_PKL}  (target={ds_pair.timepoints_ordered[-1]}, n_pcs={ds_pair.X_pca.shape[1]})")
    print(f"built {CLF_PKL}  ({len(clf_ds.ct)} cells, {len(clf_ds.ct_ordered)} classes)")

## 3. Train the local-neighbourhood probes *(only if you want `ct/*` and `recon/*`)*

The label-free metrics (§4) run without trained probes. The `concordance` / `ct_gap` groups need one
**`SpatialCTClassifierNet`**, trained on the classifier slide from §2 (`../data/abca_1.026_clf.pkl`)
so it lives in the same basis as the apparatus. This pair has **20** cell types, so override
`data.n_classes=20` (the config default is 17). Train once and reuse the single checkpoint for every
model; the old per-model checkpoints are **incompatible** (different recipe / vocabulary).

The new `recon/*` group needs a second checkpoint: a masked-centroid expression regressor. It reuses
`SpatialCTClassifierNet` with `output_dim = n_pcs`, but its dataset target is `target="expr"`, so it
predicts the centroid expression from its KNN neighbours. The checkpoint below is copied to
`../ckpts/recon_spatial.ckpt`, and §4 passes it with `--regressor`.

The cell below trains both probes on the **allocated GPU** — the kernel runs inside your SLURM job, so
`experiment=classifier/abca_spatial` / `experiment=classifier/abca_spatial_regressor` land on the
job's GPU automatically.

> **Driver note:** this needs `torch+cu126` (the node's driver is CUDA 12.7). With the current
> `torch+cu130` build it errors with *"driver too old"* — repin to cu126 and `uv sync` first.

In [ ]:
%%bash
# Runs on the job's GPU (kernel is inside the SLURM allocation). `cd ..` -> repo root so $PWD and the
# relative out-dirs match a terminal there.
cd ..

# Cell-type classifier checkpoint (monitors val/f1) -> ckpts/clf_spatial.ckpt.
uv run python -m paired_slides_eval.classifier.train experiment=classifier/abca_spatial \
    data.datamodule.data_fp=$PWD/data/abca_1.026_clf.pkl data.n_classes=20 \
    logger=tensorboard hydra.run.dir=outputs/clf_train

# Masked-centroid expression regressor checkpoint (monitors val/mse) -> ckpts/recon_spatial.ckpt.
uv run python -m paired_slides_eval.classifier.train experiment=classifier/abca_spatial_regressor \
    data.datamodule.data_fp=$PWD/data/abca_1.026_clf.pkl data.n_classes=20 \
    logger=tensorboard hydra.run.dir=outputs/recon_train

In [ ]:
%%bash
cd ../
mkdir -p ckpts
cp "$(ls -t outputs/clf_train/checkpoints/epoch_*.ckpt | head -1)" ckpts/clf_spatial.ckpt
cp "$(ls -t outputs/recon_train/checkpoints/epoch_*.ckpt | head -1)" ckpts/recon_spatial.ckpt
echo wrote ckpts/clf_spatial.ckpt
echo wrote ckpts/recon_spatial.ckpt

## 4. Run the evaluation

This is the same `python -m paired_slides_eval.evaluate` command you'd run in the terminal (a thin
wrapper over `evaluate_files`). Point every model at the **same** `data/abca_pair.pkl`; reconciliation
into the shared space is automatic, so **one command works for all four models**:

- **Expression** — gene-space cells (OT-CFM) are projected into the pickle's whitened shared PCA;
  already-reduced cells (NicheFlow) pass through (dimension auto-detect).
- **Coordinates** — `--coords auto` (default) detects raw vs already-standardised coords and maps
  them into the target frame, logging the decision. No per-model flags.

`--classifier` enables the `ct/*` groups; `--ct_real_reference fixed` makes `ct/acc_real` one
model-independent constant across the table. `--regressor` enables the `recon/*` group; by default
`recon/mse_real` also uses a fixed seeded real-target reference. The `%%bash` cells `cd ..` to the repo
root so the paths match a terminal there.

### 4a. Full metric suite → per-model CSV

Runs the whole suite for each model and writes `reports/<model>/metrics.csv` (the `--out` `metric,value`
layout). The `ct/*` groups are added when the §3 classifier exists, `recon/*` is added when the §3
regressor exists, and either probe skips cleanly otherwise.

In [ ]:
%%bash
cd ..
CKPT=ckpts/clf_spatial.ckpt
RECON=ckpts/recon_spatial.ckpt
CLF=""; [ -f "$CKPT" ] && CLF="--classifier $CKPT --ct_real_reference fixed"
REG=""; [ -f "$RECON" ] && REG="--regressor $RECON"
[ -z "$CLF" ] && echo "no classifier at $CKPT -> ct/* groups will skip"
[ -z "$REG" ] && echo "no regressor at $RECON -> recon/* group will skip"
METRIC_GROUPS="psd spd distribution c2st c2st_nn moran concordance ct_gap recon"
for m in otcfm_1025 otcfm_spatial_1025 nicheflow_cfm_unaligned nicheflow_vfm; do
    echo "===== $m ====="
    uv run python -m paired_slides_eval.evaluate \
        --target data/abca_pair.pkl \
        --generated artifacts/$m/generated.h5ad \
        --ct_key class --groups $METRIC_GROUPS $CLF $REG \
        --out reports/$m/metrics_updated.csv
done

**Optional** — collate the per-model `metrics.csv` files the CLI just wrote into one wide
comparison table (`reports/model_comparison.csv`)

In [ ]:
import glob
import os

import pandas as pd

cols = {}
for csv_path in sorted(glob.glob("../reports/*/metrics.csv")):
    name = os.path.basename(os.path.dirname(csv_path))
    cols[name] = pd.read_csv(csv_path, index_col="metric")["value"]
table = pd.DataFrame(cols).sort_index()
table.to_csv("../reports/model_comparison.csv")
print("wrote ../reports/model_comparison.csv")
table

### 4b. A single model / subset of groups → print

Drop `--out` and the CLI just prints the metrics (plus the coord auto-decision under `notes:`). Swap
the `--generated` model and `--groups` to inspect a specific metric. Available groups: `psd`, `spd`,
`distribution` (MMD/EMD), `c2st`, `c2st_nn`, `moran`, and — with `--classifier` / `--regressor` —
`concordance`, `ct_gap`, `recon`.

In [ ]:
%%bash
cd ..
python -m paired_slides_eval.evaluate \
    --target data/abca_pair.pkl \
    --generated artifacts/otcfm_spatial_1025/generated.h5ad \
    --classifier ckpts/clf_spatial.ckpt --ct_real_reference fixed \
    --regressor ckpts/recon_spatial.ckpt \
    --ct_key class --groups distribution c2st c2st_nn moran recon